In [1]:
import sys
import os
# Add parent directory to path so we can import from nwsig package
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import iisignature as ii
from aeon.datasets import load_classification
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix, r2_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.utils import shuffle

# Now import from nwsig package
from nwsig import NWSignatureClassifier, augment_with_time, suggest_truncation_level

In [5]:
classification_datasets = ["ArticularyWordRecognition",
                           "AtrialFibrillation",
                           "BasicMotions",
                           "Cricket",
                           "Epilepsy",
                           "EthanolConcentration",
                           "ERing",
                           "FaceDetection",
                           "FingerMovements",
                           "HandMovementDirection",
                           "Handwriting",
                           "Heartbeat",
                           "Libras",
                           "LSST",
                           "NATOPS",
                           "PenDigits",
                           "RacketSports",
                           "SelfRegulationSCP1",
                           "SelfRegulationSCP2",
                           "StandWalkJump",
                           "UWaveGestureLibrary"
                           ]

In [ ]:
performance = []
for dataset in classification_datasets:

    X_tr, y_tr= load_classification(dataset, split="train")
    X_te, y_te = load_classification(dataset, split="test")

    X_tr, y_tr = shuffle(X_tr, y_tr, random_state=1)  # Set random_state for reproducibility
    X_te, y_te = shuffle(X_te, y_te, random_state=1)  # Set random_state for reproducibility

    label_encoder = LabelEncoder()
    y_tr = label_encoder.fit_transform(y_tr)  # Converts strings to 0,1,2,3...
    y_te = label_encoder.fit_transform(y_te)

    print(" Shape of X = ", X_tr.shape)

    X_tr_transformed = augment_with_time(X_tr)
    X_te_transformed = augment_with_time(X_te)

    # Normalize
    scaler = MinMaxScaler()

    # Scale Train data
    X_tr_transformed_normalized = scaler.fit_transform(
        X_tr_transformed.reshape(-1, X_tr_transformed.shape[-1])
    ).reshape(X_tr_transformed.shape)

    # Scale Test data
    X_te_transformed_normalized = scaler.fit_transform(
        X_te_transformed.reshape(-1, X_te_transformed.shape[-1])
    ).reshape(X_te_transformed.shape)

    # Signature depth
    K = np.min([suggest_truncation_level(n_obs=X_tr.shape[0], length=X_tr.shape[2], dim=X_tr.shape[1], max_sig_dim_total=5e6)-1, 5])
    K = np.max([K, 1])  # Ensure K is at least 1

    # Extract initial conditions (first point of each path)
    #X0_tr = X_tr_transformed_normalized[:, 0, :]  # shape: (n_samples, n_dimensions)
    #X0_te = X_te_transformed_normalized[:, 0, :]  # shape: (n_samples, n_dimensions)
    
    X0_tr = np.zeros_like(X_tr_transformed_normalized[:, 0, :])
    X0_te = np.zeros_like(X_te_transformed_normalized[:, 0, :])

    # Fit and predict with NWSignatureClassifier
    NWclissifierCV = NWSignatureClassifier(
    input_type='paths',
    bandwidth_grid=np.linspace(0.00001, 2.0, 50),
    signature_level=K
    )
    NWclissifierCV.fit(X_tr_transformed_normalized, X0_tr, y_tr)
    Y_hat_te = NWclissifierCV.predict(X_te_transformed_normalized, X0_te)



    acc = accuracy_score(y_te, Y_hat_te)
    performance.append(acc)

    print(f"{dataset}: {acc}")

 Shape of X =  (275, 9, 144)
Fitting NW estimator with input_type='paths'
Input shape: (275, 144, 10), target shape: (275,)
Target statistics: mean=12.000000, std=7.211103, range=[0.000000, 24.000000]
Computing signatures from paths...
Starting cross-validation for bandwidth selection...
Cross-validation: Testing 50 total parameter combinations
  Signature levels: 1
  Bandwidths: 50
Testing signature level 3...
Computed signatures shape: (275, 1110)
    New best: level=3, bandwidth=1e-05, score=0.040000
    New best: level=3, bandwidth=0.040826122448979596, score=0.927273

CV Summary: 250/250 successful attempts
Best CV Accuracy: 0.927273
Cross-validation completed. Best bandwidth: 0.040826122448979596
Training completed. Final training data shape: torch.Size([275, 1110])
Using bandwidth: 0.040826122448979596, signature level: 3
ArticularyWordRecognition: 0.9566666666666667
 Shape of X =  (15, 2, 640)
Fitting NW estimator with input_type='paths'
Input shape: (15, 640, 3), target shape:

In [18]:
for name, perf in zip(classification_datasets, performance):
    print(f"{name:30s}  ->  {perf:.4f}")


ArticularyWordRecognition       ->  0.9567
AtrialFibrillation              ->  0.3333
BasicMotions                    ->  0.9000
Cricket                         ->  0.7639
Epilepsy                        ->  0.5362
EthanolConcentration            ->  0.2700
ERing                           ->  0.6889
FaceDetection                   ->  0.5142
FingerMovements                 ->  0.4900
HandMovementDirection           ->  0.2568
Handwriting                     ->  0.1388
Heartbeat                       ->  0.7268
Libras                          ->  0.7500
LSST                            ->  0.3260
NATOPS                          ->  0.7833
PenDigits                       ->  0.9254
RacketSports                    ->  0.7368
SelfRegulationSCP1              ->  0.6928
SelfRegulationSCP2              ->  0.5389
StandWalkJump                   ->  0.3333
UWaveGestureLibrary             ->  0.8344
